
# ENDS-Shortlist on Arena Human Preference 140K — corrected full experiment notebook

This notebook runs the full real-data pipeline described in the MD note:

1. Load the real `lmarena-ai/arena-human-preference-140k` parquet data.
2. Build a supported model-pair graph for `K=20` models.
3. Convert Arena outcomes into four-outcome categorical reservoirs.
4. Create pair-stratified replay/oracle splits.
5. Compute held-out oracle graph-Borda rankings.
6. Verify the exact outsider-scenario KL projection.
7. Run fixed-budget replay experiments for ENDS-Shortlist and baselines.
8. Save run-level and aggregate summaries.

Important corrections relative to the previous version:

- No synthetic fallback.
- No default `datasets.load_dataset()` call.
- The default loader downloads/reads the parquet file directly, avoiding the Hugging Face `httpx` client error.
- The projection uses the corrected scenario
  
  `Alt_{S->x} = {rho: B_x(rho) >= B_i(rho), for every i in S}`.

The default settings are real-data settings: `K=20`, budgets up to `20000`, all main baselines enabled.



## 0. Installation

Run this once in your environment, then restart the kernel:

```bash
pip install -r requirements_ends_shortlist_correct.txt
```

Recommended Python version: **Python 3.11.x**.


In [ ]:

from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make sure the local module is importable when the notebook is opened elsewhere.
PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from ends_shortlist_arena_correct import (
    ExperimentConfig,
    HF_PARQUET_URL,
    HF_MIRROR_PARQUET_URL,
    load_arena_dataframe,
    filter_arena_dataframe,
    select_dense_model_subset,
    build_graph_and_reservoirs,
    graph_diagnostics,
    outcome_distribution_frame,
    split_reservoirs,
    reservoir_counts,
    oracle_best_from_reservoirs,
    borda_frame,
    counts_to_probs,
    borda_scores,
    projection_sanity_check,
    ends_distribution,
    initialize_counts,
    choose_final_shortlist,
    run_complete_experiment,
)

print('Python:', sys.version)
print('Working directory:', PROJECT_DIR)
print('Official direct parquet URL:', HF_PARQUET_URL)
print('Mirror direct parquet URL:', HF_MIRROR_PARQUET_URL)



## 1. Full experiment configuration

Set `DATA_PATH` if you have already downloaded the parquet manually. It can be:

- a single `.parquet` file;
- a directory containing parquet files;
- a glob like `D:/Datasets/Arena140K/**/*.parquet`;
- a direct parquet URL.

If `DATA_PATH = None`, the notebook downloads the official auto-converted parquet into `CACHE_DIR` and reads it locally.


In [ ]:

# -----------------------------
# Data location
# -----------------------------
DATA_PATH = None
# Example Windows path:
# DATA_PATH = r"D:\Datasets\Arena140K\0000.parquet"
# Example Linux path:
# DATA_PATH = "/data/arena-human-preference-140k/default/train/0000.parquet"

CACHE_DIR = "./arena140k_cache"
USE_HF_MIRROR = False   # set True if huggingface.co is slow/unavailable in your network

# Optional data slices. Keep both None for the main full logged-pair target.
LANGUAGE = None         # e.g. "English" or "en" only if the column uses that label
IS_CODE = None          # True, False, or None

# -----------------------------
# Main real experiment settings
# -----------------------------
K_TARGET = 20
CANDIDATE_MODEL_POOL_SIZE = 53
MIN_PAIR_SUPPORT = 20
ORACLE_FRACTION = 0.40
REPLAY_WITH_REPLACEMENT = True

SHORTLIST_SIZE = 3
LAMBDA_BB = 0.0
ALPHA = 0.5
PI_CLIP_EPS = 1e-10

# ENDS computational mode.
# exact_candidates: exact D_{S->x} projections for all evaluated candidate shortlists.
# exact_all: exhaustive all shortlists; mathematically exhaustive but usually very slow.
NOMINATION_MODE = "exact_candidates"
DECISION_MODE = "exact_candidates"
TOP_POOL = 12
MAX_CANDIDATES = 60
RECOMPUTE_EVERY = 100
PROJECTION_MAXITER = 80
PROJECTION_TOL = 1e-8

# Full fixed-budget grid. These are total queries including the one-query-per-edge initialization.
BUDGETS = [500, 1000, 2000, 5000, 10000, 20000]
SPLIT_SEEDS = [0, 1, 2]
ALGORITHMS = [
    "ENDS-Shortlist",
    "Uniform",
    "Arena-frequency",
    "Thompson-shortlist",
    "Borda-LUCB",
    "Top1-ENDS-padding",
    "Empirical-top3",
]

OUTPUT_DIR = "./outputs_ends_shortlist_correct"
RUN_FULL_EXPERIMENT = True

cfg = ExperimentConfig(
    data_path=DATA_PATH,
    cache_dir=CACHE_DIR,
    use_hf_mirror=USE_HF_MIRROR,
    language=LANGUAGE,
    is_code=IS_CODE,
    k_target=K_TARGET,
    candidate_model_pool_size=CANDIDATE_MODEL_POOL_SIZE,
    min_pair_support=MIN_PAIR_SUPPORT,
    oracle_fraction=ORACLE_FRACTION,
    replay_with_replacement=REPLAY_WITH_REPLACEMENT,
    shortlist_size=SHORTLIST_SIZE,
    lambda_bb=LAMBDA_BB,
    alpha=ALPHA,
    pi_clip_eps=PI_CLIP_EPS,
    nomination_mode=NOMINATION_MODE,
    decision_mode=DECISION_MODE,
    top_pool=TOP_POOL,
    max_candidates=MAX_CANDIDATES,
    recompute_every=RECOMPUTE_EVERY,
    projection_maxiter=PROJECTION_MAXITER,
    projection_tol=PROJECTION_TOL,
    budgets=tuple(BUDGETS),
    split_seeds=tuple(SPLIT_SEEDS),
    algorithms=tuple(ALGORITHMS),
    output_dir=OUTPUT_DIR,
    progress=True,
)

cfg



## 2. Load the real Arena140K data

This cell is intentionally not using `datasets.load_dataset()`. The previous error came from the Hugging Face online client path; here the data is read from local parquet, downloading the parquet directly first only when necessary.


In [ ]:

t0 = time.time()
df_raw = load_arena_dataframe(
    data_path=cfg.data_path,
    cache_dir=cfg.cache_dir,
    use_hf_mirror=cfg.use_hf_mirror,
)
print(f"Loaded raw rows: {len(df_raw):,} in {time.time() - t0:.1f}s")
print("Columns:", list(df_raw.columns))
display(df_raw.head(3))
print("winner distribution:")
display(df_raw["winner"].value_counts(dropna=False).to_frame("count"))



## 3. Apply optional slice filters

For the main experiment, this should usually keep all rows.


In [ ]:

df = filter_arena_dataframe(df_raw, language=cfg.language, is_code=cfg.is_code)
print(f"Rows after filters: {len(df):,}")
print(f"Number of model_a/model_b unique values: {pd.concat([df['model_a'], df['model_b']]).nunique():,}")
if 'language' in df.columns:
    print('Top language labels:')
    display(df['language'].value_counts(dropna=False).head(10).to_frame('count'))
if 'is_code' in df.columns:
    print('is_code distribution:')
    display(df['is_code'].value_counts(dropna=False).to_frame('count'))



## 4. Select K=20 models and build the eligible pair graph

The graph includes only model pairs with at least `MIN_PAIR_SUPPORT` logged votes. If a complete 20-model graph is unavailable, the implementation uses supported graph-Borda exactly as specified in the MD.


In [ ]:

selected_models = select_dense_model_subset(
    df,
    k_target=cfg.k_target,
    min_pair_support=cfg.min_pair_support,
    candidate_model_pool_size=cfg.candidate_model_pool_size,
)
print('Selected models:')
display(pd.DataFrame({'model_idx': range(len(selected_models)), 'model': selected_models}))

graph, reservoirs = build_graph_and_reservoirs(
    df,
    selected_models=selected_models,
    min_pair_support=cfg.min_pair_support,
)

diag = graph_diagnostics(graph)
print('Graph diagnostics:')
display(pd.DataFrame([diag]))
print('Eligible edge preview:')
display(graph.to_edge_frame().sort_values('support', ascending=False).head(20))
print('Overall four-outcome distribution in eligible reservoirs:')
display(outcome_distribution_frame(reservoirs))



## 5. Pair-stratified replay/oracle split and oracle Borda ranking

The adaptive algorithms only sample from the replay reservoirs. The oracle reservoirs define the held-out Borda-best model used for success evaluation.


In [ ]:

SPLIT_SEED_FOR_DIAGNOSTICS = cfg.split_seeds[0]
replay_res, oracle_res = split_reservoirs(
    reservoirs,
    oracle_fraction=cfg.oracle_fraction,
    seed=SPLIT_SEED_FOR_DIAGNOSTICS,
)

print('Replay outcome distribution:')
display(outcome_distribution_frame(replay_res))
print('Oracle outcome distribution:')
display(outcome_distribution_frame(oracle_res))

oracle_best_idx, oracle_B, oracle_counts = oracle_best_from_reservoirs(
    graph,
    oracle_res,
    alpha=cfg.alpha,
    lambda_bb=cfg.lambda_bb,
    clip_eps=cfg.pi_clip_eps,
)
print('Oracle best model:', oracle_best_idx, graph.models[oracle_best_idx])
display(borda_frame(graph, oracle_B).head(20))



## 6. Initialize one query per eligible pair

This guarantees every pair has positive count before ENDS sampling and before KL projections.


In [ ]:

rng = np.random.default_rng(12345)
counts0 = initialize_counts(
    graph,
    replay_res,
    rng,
    with_replacement=cfg.replay_with_replacement,
)
print('Initialized query count:', int(counts0.sum()))
print('Number of eligible edges:', graph.E)
print('Min edge count after initialization:', counts0.sum(axis=1).min())
print('Initial empirical Borda ranking:')
initial_probs = counts_to_probs(counts0, prior=cfg.alpha, clip_eps=cfg.pi_clip_eps)
initial_B = borda_scores(graph, initial_probs, lambda_bb=cfg.lambda_bb)
display(borda_frame(graph, initial_B).head(10))



## 7. Exact outsider-scenario projection sanity check

This verifies the core mathematical object:

`D_{S->x}(w; pi) = inf_{rho: B_x(rho) >= B_i(rho), all i in S} sum_u w_u KL(pi_u || rho_u)`.

The output prints `B_x(rho) - B_i(rho)` constraint values. They should be nonnegative up to numerical tolerance.


In [ ]:

san = projection_sanity_check(
    graph,
    counts0,
    lambda_bb=cfg.lambda_bb,
    alpha=cfg.alpha,
    clip_eps=cfg.pi_clip_eps,
    maxiter=cfg.projection_maxiter,
    tol=cfg.projection_tol,
)
for k, v in san.items():
    print(f"{k}: {v}")



## 8. One ENDS distribution diagnostic

This runs one Estimate → Nominate → Detect → Select computation and shows the highest-probability model pairs selected by ENDS.


In [ ]:

h, info = ends_distribution(
    graph=graph,
    counts=counts0,
    rng=np.random.default_rng(2026),
    m=cfg.shortlist_size,
    lambda_bb=cfg.lambda_bb,
    alpha=cfg.alpha,
    clip_eps=cfg.pi_clip_eps,
    nomination_mode=cfg.nomination_mode,
    top_pool=cfg.top_pool,
    max_candidates=cfg.max_candidates,
    projection_maxiter=cfg.projection_maxiter,
    projection_tol=cfg.projection_tol,
)
print('ENDS diagnostic info:')
for k, v in info.items():
    print(f"{k}: {v}")
print('H sums to:', h.sum())
edge_prob = graph.to_edge_frame().copy()
edge_prob['H_probability'] = h
edge_prob = edge_prob.sort_values('H_probability', ascending=False)
display(edge_prob.head(20))



## 9. Run the full fixed-budget replay experiment

This is the real experiment cell. It uses the full Arena data, K=20 graph, all configured budgets, and all configured algorithms.

Runtime depends heavily on `MAX_CANDIDATES`, `RECOMPUTE_EVERY`, number of seeds, and whether you choose `exact_all`. The default is intended to finish on a workstation while still using exact scenario projections for evaluated shortlists.


In [ ]:

if RUN_FULL_EXPERIMENT:
    t0 = time.time()
    results = run_complete_experiment(cfg)
    print(f"Full experiment finished in {(time.time() - t0)/60:.2f} minutes")
    print('Output directory:', results['output_dir'])
    run_df = results['run_df']
    agg_df = results['agg_df']
    oracle_df = results['oracle_df']
    trace_df = results['trace_df']
else:
    print('RUN_FULL_EXPERIMENT is False; skipping full experiment.')
    results = None
    run_df = pd.DataFrame()
    agg_df = pd.DataFrame()
    oracle_df = pd.DataFrame()
    trace_df = pd.DataFrame()



## 10. Inspect run-level and aggregate results


In [ ]:

print('Run-level summary shape:', run_df.shape)
display(run_df.head(20))
print('Aggregate summary:')
display(agg_df)



## 11. Plot success rate vs budget


In [ ]:

if not agg_df.empty:
    plt.figure(figsize=(10, 6))
    for alg, sub in agg_df.groupby('algorithm'):
        sub = sub.sort_values('budget')
        plt.plot(sub['budget'], sub['success_rate'], marker='o', label=alg)
    plt.xscale('log')
    plt.ylim(-0.02, 1.02)
    plt.xlabel('Total replay budget')
    plt.ylabel('Success rate: oracle best in returned shortlist')
    plt.title('ENDS-Shortlist Arena140K fixed-budget replay')
    plt.legend(loc='best')
    plt.grid(True, which='both', alpha=0.3)
    plt.show()
else:
    print('No aggregate results to plot.')



## 12. Inspect ENDS trace

This is useful for verifying which shortlist and outsider pitfall ENDS detects over time.


In [ ]:

print('Trace shape:', trace_df.shape)
if not trace_df.empty:
    display(trace_df.head(50))
    cols = [c for c in ['n', 'shortlist_models', 'outsider_model', 'scenario_D', 'entropy_H', 'projection_violation', 'elapsed_sec'] if c in trace_df.columns]
    display(trace_df[cols].tail(30))
else:
    print('No trace rows saved.')



## 13. Saved files


In [ ]:

outdir = Path(cfg.output_dir)
print('Output directory:', outdir.resolve())
for p in sorted(outdir.glob('*')):
    print(p.name, f"{p.stat().st_size/1024:.1f} KB")



## 14. Optional exhaustive verification cell

This cell is off by default. Turn it on only for a small budget/seed or after the main pipeline works. `exact_all` enumerates every size-3 shortlist and is much slower.


In [ ]:

RUN_EXHAUSTIVE_VERIFICATION = False

if RUN_EXHAUSTIVE_VERIFICATION:
    cfg_ex = ExperimentConfig(**{**cfg.__dict__})
    cfg_ex.nomination_mode = 'exact_all'
    cfg_ex.decision_mode = 'exact_all'
    cfg_ex.budgets = (500,)
    cfg_ex.split_seeds = (0,)
    cfg_ex.algorithms = ('ENDS-Shortlist',)
    cfg_ex.output_dir = './outputs_ends_shortlist_exhaustive_check'
    cfg_ex.recompute_every = 200
    res_ex = run_complete_experiment(cfg_ex)
    display(res_ex['agg_df'])
else:
    print('Exhaustive verification skipped. Set RUN_EXHAUSTIVE_VERIFICATION=True to run it.')
